In [0]:
from pyspark.sql.functions import col, row_number, to_timestamp, current_timestamp
from pyspark.sql.window import Window

In [0]:

patient_deduplicate = spark.sql('''select active, address,	birthDate, communication, contact, extension, gender, generalPractitioner, id,	identifier,	managingOrganization,maritalStatus,meta,name,resourceType,	telecom, text,file_path,	extraction_timestamp,api_url_or_params from (select active, address,	birthDate, communication, contact, extension, gender, generalPractitioner, id,	identifier,	managingOrganization,maritalStatus,meta,name,resourceType,	telecom, text,file_path,extraction_timestamp,api_url_or_params, row_number() over (partition by id order by meta.lastUpdated desc) as rn from workspace.default.bronze_patient where id is not null) where rn=1''')

patient_deduplicate = patient_deduplicate.withColumn("silver_processed_timestamp",current_timestamp())
patient_deduplicate.write.format('delta').mode('overwrite').saveAsTable('workspace.default.silver_patient')

In [0]:
%sql
-- select count(id), lastUpdated from (select id,name,identifier,meta.lastUpdated as lastUpdated from workspace.default.bronze_patient where meta.lastUpdated  = '2026-08-10T20:53:16.775-04:00') group by LastUpdated having count(id)>1